In [1]:
import pandas as pd
import numpy as np
import math
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import datetime
from scipy import stats
import warnings
import datetime
import os
from datetime import datetime
warnings.filterwarnings('ignore')

In [2]:
# 0 -- 6 month
# 1 -- 2 year
# 2 -- 5 year
# 3 -- 10 year

rate_reg = 3

In [3]:
load_path = r'\\osiride-fs\group\main\891af\private\Area_MF\Varie\Poli_Venturi\Data\Database_Regression.xlsx'
data = pd.read_excel(load_path,index_col=[0],parse_dates=[0])
data = data.loc[datetime(2000,1,1):datetime(2025,1,1)]
data.reset_index(inplace=True,drop=False)

load_path = r'\\osiride-fs\group\main\891af\private\Area_MF\Varie\Poli_Venturi\Data\Surprises_Reduced.xlsx'
surprises = pd.read_excel(load_path,index_col=[0],parse_dates=[0])
surprises = surprises.loc[datetime(2000,1,1):datetime(2025,1,1)]
surprises.reset_index(inplace=True,drop=False)

mergedData = data.merge(surprises,on='Date',how='outer')

surprises = mergedData.loc[:,surprises.columns]
surprises.fillna(0,inplace=True)

mergedData = data.merge(surprises,on='Date',how='outer')
mergedData.set_index('Date',drop=True,inplace=True)

mergedData = mergedData.loc[datetime(2000,1,1):datetime(2025,1,1),[
    'SwapESTR6Md','SwapESTR2Yd','SwapESTR5Yd','SwapESTR10Yd',
    'SwapESTR6Mm','SwapESTR2Ym','SwapESTR5Ym','SwapESTR10Ym',
    'OASd','EASurprise','USSurprise',
    'M3','CPI EA','CPI FR','CPI ES','CPI DE','CPI IT','Industrial Production','Retail Sales DE','IFO Business Survey','ZEW Expectations','ZEW Current Conditions',
    'CPI US', 'Core CPI US', 'PPI US','Core PPI US', 'Employment Cost Index', 'Average Hourly Earnings',
    'GDP', 'Durables', 'Retail Sales', 'Retail Ex. Auto','Unemployment','Non-Farm Payrolls',
    'DE GDP', 'Factory Orders', 'IFO Current Conditions', 'IFO Expectations', 'GDP FR', 'Unemployment Claims'
]]
mergedData.dropna(inplace=True)
    
diff_rates_clean = mergedData.loc[:,['SwapESTR6Md','SwapESTR2Yd','SwapESTR5Yd','SwapESTR10Yd']]

z = np.abs(stats.zscore(diff_rates_clean))
threshold = 10

outliers = pd.concat([diff_rates_clean[z.SwapESTR6Md>threshold],diff_rates_clean[z.SwapESTR2Yd>threshold],
                     diff_rates_clean[z.SwapESTR5Yd>threshold],diff_rates_clean[z.SwapESTR10Yd>threshold],
                     diff_rates_clean.loc[datetime(2023,3,13):datetime(2023,3,15)]],axis=0)

outliers.reset_index(inplace=True,drop=False)
outliers.drop_duplicates(subset='Date',inplace=True)
outliers.sort_values(by='Date',inplace=True,ignore_index=True)
outliers.set_index('Date',inplace=True,drop=True)

diff_rates_clean.drop(outliers.index,inplace=True,axis=0)

idx = diff_rates_clean.index

mergedData = mergedData.loc[idx]

In [4]:
ratevol_estr = mergedData.loc[:,['SwapESTR6Md','SwapESTR2Yd','SwapESTR5Yd','SwapESTR10Yd']].fillna(method='ffill').rolling(window=5,closed='left').std()
ratevol_estr = (ratevol_estr - ratevol_estr.mean())/ratevol_estr.std()
for j in ratevol_estr.columns:
    ratevol_estr.rename(columns={j: j.replace('Yd','Yv')},inplace=True)
    ratevol_estr.rename(columns={j: j.replace('Md','Mv')},inplace=True)

diff_rates_estr = mergedData.loc[:,['SwapESTR6Md','SwapESTR2Yd','SwapESTR5Yd','SwapESTR10Yd']]

df_momentum_estr = mergedData.loc[:,['SwapESTR6Mm','SwapESTR2Ym','SwapESTR5Ym','SwapESTR10Ym']]
df_momentum_estr.fillna(method='ffill',inplace=True)
df_momentum_estr = (df_momentum_estr - df_momentum_estr.mean())/df_momentum_estr.std()

diff_oas = mergedData.loc[:,['OASd']]
diff_oas = (diff_oas - diff_oas.mean())/diff_oas.std()

df_surprise_ea = mergedData.loc[:,['EASurprise']]
df_surprise_ea = (df_surprise_ea - df_surprise_ea.mean())/df_surprise_ea.std()

df_surprise_us = mergedData.loc[:,['USSurprise']]
df_surprise_us = (df_surprise_us - df_surprise_us.mean())/df_surprise_us.std()

df_surprise = mergedData.loc[:,surprises.columns[1:].to_list()]
df_surprise.fillna(0,inplace=True)

df_surprise_cat = pd.DataFrame()

df_surprise_cat['EA'] = df_surprise.loc[:,['M3','CPI EA','CPI FR','CPI ES','CPI DE','CPI IT',
                                           'Industrial Production','Retail Sales DE',
                                           'IFO Business Survey','ZEW Expectations','ZEW Current Conditions']].sum(axis=1)

df_surprise_cat['USInf'] = df_surprise.loc[:,['CPI US', 'Core CPI US', 'PPI US',
                                                 'Core PPI US', 'Employment Cost Index', 'Average Hourly Earnings']].sum(axis=1)
df_surprise_cat['USGrw'] = df_surprise.loc[:,['GDP', 'Durables', 'Retail Sales', 'Retail Ex. Auto']].sum(axis=1)
df_surprise_cat['USLab'] = df_surprise.loc[:,['Unemployment','Non-Farm Payrolls']].sum(axis=1)
df_surprise_cat['US'] = df_surprise_cat.loc[:,['USInf','USGrw','USLab']].sum(axis=1)

### Full Sample

In [5]:
df_reg = pd.concat([diff_rates_estr.iloc[:,rate_reg],
                    df_surprise_cat.loc[:,['EA']],df_surprise_cat.loc[:,['US']],
                    df_momentum_estr.iloc[:,rate_reg],ratevol_estr.iloc[:,rate_reg],diff_oas],axis=1)
    
Y = df_reg.iloc[:,[0]]
X = df_reg.iloc[:,1:]
X = sm.add_constant(X)

model = sm.OLS(Y, X, missing='drop').fit(cov_type='HAC',cov_kwds={'maxlags':math.ceil(0.75*len(Y)**(1/3))})
display(model.summary())

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           SwapESTR10Yd   R-squared:                       0.163
Model:                            OLS   Adj. R-squared:                  0.162
Method:                 Least Squares   F-statistic:                     167.0
Date:                Sun, 08 Feb 2026   Prob (F-statistic):          3.82e-167
Time:                        12:32:19   Log-Likelihood:                -17156.
No. Observations:                6229   AIC:                         3.432e+04
Df Residuals:                    6223   BIC:                         3.437e+04
Df Model:                           5                                         
Covariance Type:                  HAC                                         
================================================================================
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const           -0.0536      0.037     -1.466      0.143      -0.125       0.018
EA               0.3359      0.086      3.907      0.000       0.167       0.504
US               0.1692      0.059      2.889      0.004       0.054       0.284
SwapESTR10Ym     1.2023      0.052     22.974      0.000       1.100       1.305
SwapESTR10Yv     0.0956      0.069      1.382      0.167      -0.040       0.231
OASd            -1.0091      0.137     -7.389      0.000      -1.277      -0.741
==============================================================================
Omnibus:                      685.279   Durbin-Watson:                   1.993
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             6350.258
Skew:                           0.038   Prob(JB):                         0.00
Kurtosis:                       7.946   Cond. No.                         1.27
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 14 lags and without small sample correction
"""